# Iter-3 Phase 3B — Main PyMC fit

Hierarchical Lognormal-Poisson MCMC over the curated analog-mission incidence corpus. Reads `research/evidence_extracted/incidence_rates.csv` (Diego-curated rows only, per Task 37), samples NUTS with `random_seed=0xC0FFEE` for byte-identical reproducibility, asserts convergence per [G12] / [M18] rules, and exports `src/risk/priors.json`.

**Vulnerability_beta / worst_case_prob_q / lost-day distributions are NOT fit here** — those are hand-elicited by Diego at Task 41 from the evidence tables, then merged into the same JSON.

Reproducibility contract: re-running this notebook from a clean kernel must produce `priors.json` byte-identical to the previous run. The `iter3_imm_fit.trace.nc` checkpoint is non-negotiable — if any downstream cell crashes, we do not re-sample.

In [ ]:
# Cell 1 — imports + load CSV
import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
from pathlib import Path

REPO = Path.cwd().parent
df = pd.read_csv(REPO / "research/evidence_extracted/incidence_rates.csv")
assert (df["extracted_by"] == "Diego").all(), "All rows must be Diego-accepted (see Task 37)"
df["condition_idx"] = pd.Categorical(df["condition_id"]).codes
df["mission_idx"] = pd.Categorical(df["mission_type"]).codes
n_conditions = df["condition_id"].nunique()
n_missions = df["mission_type"].nunique()
print(f"{len(df)} rows, {n_conditions} conditions, {n_missions} missions")

In [ ]:
# Cell 2 — model definition (Iter-3 spec §3.6, hierarchical extension)
with pm.Model(coords={"condition": df["condition_id"].unique(),
                       "mission": df["mission_type"].unique(),
                       "obs": np.arange(len(df))}) as imm_analog:
    mu = pm.Normal("mu_lograte", mu=0, sigma=10, dims="condition")
    tau = pm.HalfCauchy("tau_lograte", beta=2.5, dims="condition")

    log_lambda = pm.Normal(
        "log_lambda",
        mu=mu[df["condition_idx"].values],
        sigma=tau[df["condition_idx"].values],
        dims="obs",
    )
    rate = pm.math.exp(log_lambda) * df["person_days"].values
    pm.Poisson("events_obs", mu=rate, observed=df["events"].values, dims="obs")

In [ ]:
# Cell 3 — sample (with non-negotiable .nc checkpoint)
with imm_analog:
    trace = pm.sample(
        draws=4000, tune=2000, chains=4,
        target_accept=0.95, random_seed=0xC0FFEE,
        return_inferencedata=True,
    )
trace.to_netcdf(REPO / "notebooks/iter3_imm_fit.trace.nc")
print("checkpoint saved")

In [ ]:
# Cell 4 — convergence assertions ([G12] / [M18] rules)
summary = az.summary(trace, var_names=["mu_lograte", "tau_lograte"])
assert (summary["r_hat"] < 1.01).all(), summary[summary["r_hat"] >= 1.01]
assert (summary["ess_bulk"] > 1000).all(), summary[summary["ess_bulk"] <= 1000]
print("converged: r-hat <1.01, ESS >1000 on all parameters")

In [ ]:
# Cell 5 — export priors.json (mu/tau MCMC results only; β + q + lost-days at Task 41)
import json

priors = {
    "model_version": "iter3-v1",
    "fitted_at": pd.Timestamp.utcnow().isoformat(),
    "pyMC_version": pm.__version__,
    "r_hat_max": float(summary["r_hat"].max()),
    "ess_min": int(summary["ess_bulk"].min()),
    "conditions": {},
}

for cond in df["condition_id"].unique():
    priors["conditions"][cond] = {
        "missions": {},
        "vulnerability_beta": {},          # Task 41 (hand-elicited)
        "worst_case_prob_q": None,         # Task 41
        "treated_lost_days_mean": None,    # Task 41
        "untreated_lost_days_mean": None,  # Task 41
    }
    for miss in df["mission_type"].unique():
        mask = (df["condition_id"] == cond) & (df["mission_type"] == miss)
        if not mask.any():
            continue
        obs_idx = np.where(mask)[0]
        # Average across observations for this (cond, miss) cell
        samples = trace.posterior["log_lambda"].values[..., obs_idx].mean(axis=-1).flatten()
        # Thin to 1000 samples per (condition, mission)
        thinned = samples[:: max(1, len(samples) // 1000)][:1000]
        priors["conditions"][cond]["missions"][miss] = {
            "log_lambda_samples": thinned.tolist(),
            "mean_log_lambda": float(thinned.mean()),
            "sd_log_lambda": float(thinned.std()),
        }

(REPO / "src/risk/priors.json").write_text(json.dumps(priors, indent=2))
print(f"wrote {(REPO/'src/risk/priors.json').stat().st_size / 1024:.1f} KB")

## Next: Task 41 — hand-elicit β, q, lost-days

Open `src/risk/priors.json` and fill the four placeholder fields for each condition (`vulnerability_beta`, `worst_case_prob_q`, `treated_lost_days_mean`, `untreated_lost_days_mean`) using the per-condition justifications in `research/imm_sources/_beta_elicitation_audit.md`. The audit document references the evidence-table effect sizes from Phase 0 (Tasks A3/A4/A5 outputs at `research/evidence_tables/`).

## Next: Task 42 — JAGS cross-check

Run `notebooks/iter3_jags_crosscheck.ipynb` on a 4-conditions × 3-missions subset and confirm NUTS posterior summaries match Gibbs to within MC error (< 5%, per [G12]).